### Эксперимент 2. Код для Mistral

In [ ]:
pip install mistral_inference

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

mistral_models_path = Path.home().joinpath('mistral_models', 'Nemo-Instruct')
mistral_models_path.mkdir(parents=True, exist_ok=True)

snapshot_download(repo_id="mistralai/Mistral-Nemo-Instruct-2407", allow_patterns=["params.json", "consolidated.safetensors", "tekken.json"], local_dir=mistral_models_path)


In [ ]:
from mistral_inference.transformer import Transformer
from mistral_inference.generate import generate

from mistral_common.tokens.tokenizers.mistral import MistralTokenizer
from mistral_common.protocol.instruct.messages import UserMessage
from mistral_common.protocol.instruct.request import ChatCompletionRequest

from transformers import AutoTokenizer, AutoModelForCausalLM

Единая функция

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch

In [ ]:
def process_derivation_csv(
    input_csv_path: str,
    output_csv_path: str = "derivation_results.csv",
    embeddings_dir: str = "embeddings_npy",
    max_input_length: int = 256,
    max_new_tokens: int = 16,
    do_sample: bool = True,
    temperature: float = 0.35,
    top_p: float = 0.95,
    use_chat_template: bool = True,
):
    """
    Читает CSV со столбцами:
        - verb
        - definition

    Для каждой строки:
        1. генерирует дериват
        2. считает embedding по definition
        3. сохраняет embedding в .npy
        4. записывает результат в таблицу

    Возвращает pandas датафрейм.
    """

    Path(embeddings_dir).mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(input_csv_path)

    required_cols = {"verb", "definition"}
    missing = required_cols - set(df.columns)

    def safe_filename(text: str) -> str:
        return (
            str(text)
            .replace("/", "_")
            .replace("\\", "_")
            .replace(" ", "_")
            .replace(":", "_")
            .replace('"', "")
            .replace("'", "")
        )

    def generate_derivation(verb: str, definition: str) -> str:
        prompt = (
            f'В русском появился новый глагол "{verb}", который значит "{definition}".\n'
            f"Как одним словом назвать человека, который это делает?\n"
            f"Ответь только производным от глагола существительным."
        )

        if use_chat_template and hasattr(tokenizer, "apply_chat_template"):
            messages = [{"role": "user", "content": prompt}]
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        else:
            text = prompt

        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_input_length
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        gen_kwargs = dict(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )

        if do_sample:
            gen_kwargs.update(
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
            )
        else:
            gen_kwargs.update(do_sample=False)

        with torch.no_grad():
            outputs = model.generate(**gen_kwargs)

        generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        result = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        return result

    def mean_pool_last_hidden(text: str) -> torch.Tensor:
        enc = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=64,
        )
        enc = {k: v.to(model.device) for k, v in enc.items()}

        with torch.no_grad():
            out = model(
                **enc,
                output_hidden_states=True,
                return_dict=True,
                use_cache=False,
            )

        last_hidden = out.hidden_states[-1]
        mask = enc["attention_mask"].unsqueeze(-1)

        pooled = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return pooled[0].detach().cpu()

    def save_embedding(embedding: torch.Tensor, verb: str) -> str:
        safe_verb = safe_filename(verb)
        npy_path = os.path.join(embeddings_dir, f"{safe_verb}_mistral.npy")

        if embedding.dtype == torch.bfloat16:
            embedding = embedding.float()

        np.save(npy_path, embedding.numpy())
        return npy_path

    results = []

    for idx, row in df.iterrows():
        verb = str(row["verb"]).strip()
        definition = str(row["definition"]).strip()

        print(f"\n[{idx + 1}/{len(df)}] Обрабатываю: {verb} — {definition}")

        derivation = generate_derivation(verb, definition)

        embedding = mean_pool_last_hidden(definition)
        embedding_path = save_embedding(embedding, verb)
        embedding_dim = int(embedding.shape[0])

        print(f"Дериват: {derivation}")
        print(f"Эмбеддинг: {embedding_path}")

        results.append({
            "verb": verb,
            "definition": definition,
            "derivation": derivation,
            "embedding_path": embedding_path,
            "embedding_dim": embedding_dim,
        })

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)
    results_df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

    print(f"\n Эмбеддинг в: {output_csv_path}")
    return results_df

In [ ]:
results_df = process_derivation_csv(
    input_csv_path="verbs.csv",
    output_csv_path="results.csv",
    embeddings_dir="embeddings_mistral_npy",
    do_sample=True,
    temperature=0.35,
    top_p=0.95,
    use_chat_template=True,
)